## Data

Original CSV/JPG files stay unchanged. Exact SHA-256 repeats are excluded; dHash candidates are retained and grouped.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "data/train.csv").exists():
    root = root.parent
sys.path.insert(0, str(root))
from cnn.near_duplicate_audit import LABELS, find

labels = LABELS
train = pd.read_csv(root / "data/train.csv")
test = pd.read_csv(root / "data/test.csv")
print("Original train:", len(train), "Test:", len(test))
assert train.image_id.is_unique
assert train[labels].isin([0, 1]).all().all()
display(train.head())

Original train: 12260 Test: 18673


,image_id,Cat,Dog,Moris,Motya,Biatrix
0,7e17a6e589f9c891857b32bbb3652d19b7d5a215,0,0,0,0,0
1,7ad97bd6c7ef98373f43f2bbd1e41b2bbff4687e,1,0,0,0,0
2,d29be872d012d8766c5942847d1d721788fd9511,0,1,0,0,0
3,bec7727bfc721629997d782ec020ee555990128f,0,1,0,0,0
4,58bb7362153d945f48721004bc8fb2ee73a855b0,0,1,0,0,0


In [2]:
counts = train[labels].sum()
percentages = counts / len(train) * 100

distribution = pd.DataFrame({
    "Count": counts,
    "Percent": percentages.round(2)
})

display(distribution)

print("Background:", (train[labels].sum(axis=1) == 0).sum())
print("Multiple labels:", (train[labels].sum(axis=1) > 1).sum())

,Count,Percent
Cat,5077,41.41
Dog,5025,40.99
Moris,56,0.46
Motya,42,0.34
Biatrix,41,0.33


Background: 2162
Multiple labels: 125


## Metrics

Macro-F1 weights all five classes equally and ignores true negatives. It is threshold-dependent and unstable for rare classes. With `zero_division=0`, undefined F1 is zero; an absent class does not demonstrate recognition.

Also report per-class precision/recall/F1 and support. Rare Macro-F1 averages Moris, Motya and Biatrix. Independent sigmoid outputs allow multiple labels; background has none.

Average precision (AP) summarizes the PR curve without a threshold, useful for imbalance. It differs from trapezoidal PR-AUC; show NaN when no positives exist.

In [3]:
from sklearn.metrics import precision_recall_fscore_support, f1_score

y_true = train[labels].to_numpy()
y_pred = np.zeros_like(y_true)
y_pred[:, :2] = 1
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0)
print("Illustration only: always predict Cat and Dog, not a trained model")
display(pd.DataFrame({"precision": precision, "recall": recall, "F1": f1,
                      "support": support}, index=labels))
print("Macro-F1:", round(f1.mean(), 4), "Rare Macro-F1:", round(f1[2:].mean(), 4))
assert f1_score(np.zeros((2, 5)), np.zeros((2, 5)),
                average="macro", zero_division=0) == 0

Illustration only: always predict Cat and Dog, not a trained model


,precision,recall,F1,support
Cat,0.414111,1.0,0.585684,5077
Dog,0.409869,1.0,0.581429,5025
Moris,0.000000,0.0,0.000000,56
Motya,0.000000,0.0,0.000000,42
Biatrix,0.000000,0.0,0.000000,41


Macro-F1: 0.2334 Rare Macro-F1: 0.0


## Validation

Five-fold multilabel stratification on connected duplicate groups, seed 45. Keep one record per exact SHA-256 group and put every dHash candidate group in one fold. Move only singleton background images to balance fold sizes. `excluded` records never enter train or validation.

In [4]:
matches = pd.read_csv(root / "outputs/train_duplicate_audit.csv")
print("Exact SHA-256 pairs:", int(matches.sha256_equal.sum()))
print("All dHash pairs (including exact):", len(matches))
print("Non-exact candidates:", int(matches.needs_review.sum()))
print("Connected groups:", matches.group_id.nunique())
print("Pairs with label conflicts:", int(matches.label_conflict.sum()))
print("Review every non-exact candidate; start with distance 4:")
display(matches.loc[matches.needs_review & matches.hash_distance.eq(4),
                    ["group_id", "image_id_a", "image_id_b", "labels_a", "hash_distance"]])
print("Visual preview: .venv/bin/python cnn/near_duplicate_audit.py --show-group ID")

Exact SHA-256 pairs: 5
All dHash pairs (including exact): 163
Non-exact candidates: 158
Connected groups: 163
Pairs with label conflicts: 0
Review every non-exact candidate; start with distance 4:


,group_id,image_id_a,image_id_b,labels_a,hash_distance
76,77,523b71a57a66410468b6b245387f70d642c84468,f848d64861c31b5d896102f8d036b1120b52ad35,Background,4
105,106,75175e3e74cd0cf7ec863bb06fbe0f3e55b363b8,7f6b216454d45b9506fb011ddcbaee25f4c5a6e7,Background,4
114,115,7fab78144df0f9acb3e06f3bbbc73dd32a86eba8,ac451c4bae4af1c2a79695651e8837f3dedb5956,Dog|Biatrix,4
145,146,addfc4a24295235e859c9b868f0444e8a01c57a3,f5bf4b4e4b82a397afdf52c4a06b1799fad948bf,Background,4


Visual preview: .venv/bin/python cnn/near_duplicate_audit.py --show-group ID


In [5]:
parent = {image_id: image_id for image_id in train.image_id}
exact_parent = parent.copy()
by_id = train.set_index("image_id")

for row in matches.itertuples(index=False):
    a, b = row.image_id_a, row.image_id_b
    parent[find(parent, b)] = find(parent, a)
    if row.sha256_equal:
        assert by_id.loc[a, labels].equals(by_id.loc[b, labels]), "Exact label conflict"
        exact_parent[find(exact_parent, b)] = find(exact_parent, a)

train["group"] = [find(parent, image_id) for image_id in train.image_id]
train["exact_group"] = [find(exact_parent, image_id) for image_id in train.image_id]
train["excluded"] = train.image_id.ne(train.exact_group)

In [6]:
print("Excluded exact duplicate records:", int(train.excluded.sum()))
display(train.loc[train.excluded, ["image_id", "exact_group"]])
kept = train.loc[~train.excluded]
assert kept.groupby("exact_group").size().eq(1).all()
assert kept.exact_group.nunique() == train.exact_group.nunique()
display(pd.DataFrame({"original positives": train[labels].sum(),
                      "active positives": kept[labels].sum(),
                      "active percent": (100 * kept[labels].mean()).round(2)}))
print("Original images:", len(train), "Active images:", len(kept))
print("Active background:", int(kept[labels].sum(axis=1).eq(0).sum()))
print("Active multilabel:", int(kept[labels].sum(axis=1).gt(1).sum()))

Excluded exact duplicate records: 5


,image_id,exact_group
208,dc59ae12ac959ef428584eb4cc35876a2dfb5e77,3fcf3dc4635a80f65b54ecd39df4aacec3356992
532,5614ff0793569336073aa6cb231a28848b55fc3c,3b07697e27a527755a4216bf8055a3b79b68f104
6207,fd53cc3b076b0a8057ba01a2603ae379c01e9e8d,3e0f768fb6189230a9e80c6ef97ea726c56f48cb
7765,89d48c6c5391c5ff1ed6d76909a11ca80687641b,16dad32b1bfd8c4129a68cdf926374aa219861bf
12207,6456d87e01e810c86f96ad977e1e8d9df4591862,1b51b41f6a312a1e620f19c95672bc1f53444fcd


,original positives,active positives,active percent
Cat,5077,5074,41.40
Dog,5025,5023,40.99
Moris,56,56,0.46
Motya,42,42,0.34
Biatrix,41,41,0.33


Original images: 12260 Active images: 12255
Active background: 2162
Active multilabel: 125


In [7]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

groups = kept.groupby("group")[labels].sum()
cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=45)
group_folds = pd.Series(-1, index=groups.index)

for fold, (_, val_idx) in enumerate(cv.split(groups, groups.gt(0))):
    group_folds.iloc[val_idx] = fold

train["fold"] = train.group.map(group_folds).astype(int)

In [8]:
single = train.groupby("group").image_id.transform("size").eq(1)
background = train[labels].sum(axis=1).eq(0)

while True:
    sizes = train.loc[~train.excluded].groupby("fold").size()
    if sizes.max() - sizes.min() <= 1:
        break
    source, target = sizes.idxmax(), sizes.idxmin()
    candidates = train.index[single & background & train.fold.eq(source)]
    assert len(candidates), "No singleton background image available"
    train.loc[candidates[0], "fold"] = target

assert train.groupby("group").fold.nunique().eq(1).all()
kept = train.loc[~train.excluded]
summary = kept.groupby("fold")[labels].sum()
summary.insert(0, "images", kept.groupby("fold").size())
assert (summary.max() - summary.min()).le(1).all()
display(summary)

,images,Cat,Dog,Moris,Motya,Biatrix
fold,,,,,,
0,2451,1014,1004,11,8,8
1,2451,1015,1005,11,9,8
2,2451,1015,1005,11,8,9
3,2451,1015,1004,12,9,8
4,2451,1015,1005,11,8,8


In [9]:
folds = train[["image_id", "fold", "excluded"]].copy()
assert folds.image_id.is_unique
assert folds.fold.isin(range(5)).all()
assert folds.excluded.isin([True, False]).all()
folds.to_csv(root / "folds.csv", index=False)
print("Saved:", len(folds), "records; active:", int((~folds.excluded).sum()))

Saved: 12260 records; active: 12255


In [10]:
saved = pd.read_csv(root / "folds.csv")
original = pd.read_csv(root / "data/train.csv")
assert saved.image_id.is_unique
assert set(saved.image_id) == set(original.image_id)
pd.testing.assert_frame_equal(saved, folds)

check = train[["image_id", "group", "exact_group"]].merge(
    saved, on="image_id", validate="one_to_one")
cross_fold_groups = check.groupby("group").fold.nunique().gt(1).sum()
kept_counts = (~check.excluded).groupby(check.exact_group).sum()
assert cross_fold_groups == 0
assert kept_counts.eq(1).all()
assert saved.fold.isin(range(5)).all()
assert saved.excluded.isin([True, False]).all()
from cnn.loader import load_train_folds

active = load_train_folds()
seen = []
for fold in range(5):
    training = active.loc[active.fold.ne(fold)]
    validation = active.loc[active.fold.eq(fold)]
    assert not set(training.image_id).intersection(validation.image_id)
    assert not training.excluded.any() and not validation.excluded.any()
    seen.extend(validation.image_id)
assert len(seen) == len(set(seen)) == len(active)
assert set(seen) | set(saved.loc[saved.excluded, "image_id"]) == set(original.image_id)
print("Leakage check passed:", int(cross_fold_groups), "cross-fold groups;",
      len(active), "active;", int(saved.excluded.sum()), "excluded")

Leakage check passed: 0 cross-fold groups; 12255 active; 5 excluded


## CV and Public Score

Public Score uses only part of the test set. Rare-class sampling and distribution shifts can differ from CV. Threshold tuning is exploratory; test labels and Public Score are not used for selection. Grouping limits detected duplicate leakage, but unknown source relations may remain.

AI disclosure: OpenAI Codex assisted with code, explanations and local checks.